<a href="https://colab.research.google.com/github/aizaaziz/aizaaziz-DataScience-GenAI-Submissions/blob/main/Copy_of_6_02_DNN_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [ ]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  1


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [ ]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]]),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142., 155., 225.,  59

We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [ ]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [ ]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [ ]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [ ]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 27720.5527
Epoch [10/100], Loss: 32262.4785
Epoch [10/100], Loss: 29781.5352
Epoch [10/100], Loss: 27116.1113
Epoch [10/100], Loss: 30946.4219
Epoch [10/100], Loss: 32000.8945
Epoch [10/100], Loss: 28854.0488
Epoch [10/100], Loss: 22306.1797
Epoch [20/100], Loss: 31268.127
Epoch [20/100], Loss: 35332.7227
Epoch [20/100], Loss: 30691.7969
Epoch [20/100], Loss: 27171.4824
Epoch [20/100], Loss: 28346.2188
Epoch [20/100], Loss: 26804.4395
Epoch [20/100], Loss: 27897.1367
Epoch [20/100], Loss: 33894.7305
Epoch [30/100], Loss: 25894.5449
Epoch [30/100], Loss: 33843.8867
Epoch [30/100], Loss: 29589.3594
Epoch [30/100], Loss: 28684.4551
Epoch [30/100], Loss: 32795.6094
Epoch [30/100], Loss: 30991.9141
Epoch [30/100], Loss: 25239.3945
Epoch [30/100], Loss: 26035.5527
Epoch [40/100], Loss: 28066.7793
Epoch [40/100], Loss: 32021.6074
Epoch [40/100], Loss: 29823.9922
Epoch [40/100], Loss: 27365.3691
Epoch [40/100], Loss: 31305.4395
Epoch [40/100], Loss: 25528.1543
Epoch [40/1

We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:

In [ ]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse_new = np.mean(mse_values)
print(f"Average MSE on test set after 1000 epochs: {avg_mse_new}")
print(f"Previous Average MSE after 100 epochs: {avg_mse}")

Average MSE on test set: 18624.20068359375


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [ ]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,29.978760,219.0
1,28.446333,70.0
2,29.705957,202.0
3,36.936768,230.0
4,28.788906,111.0
...,...,...
84,25.766422,153.0
85,23.648552,98.0
86,21.768284,37.0
87,22.518467,63.0


Side-by-side, they don't look great. Can you improve them?

<br><br>

## EXERCISE #1
Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?




In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

epochs = 1000 # Increased to 1000 epochs

for epoch in range(epochs):
  for inputs, targets in train_loader:
    inputs, targets = inputs.to(device), targets.to(device)
    model.to(device)
    model.train()
    optimiser.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimiser.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/1000], Loss: 19649.5117
Epoch [10/1000], Loss: 17324.4043
Epoch [10/1000], Loss: 22657.25
Epoch [10/1000], Loss: 23479.9746
Epoch [10/1000], Loss: 15541.7871
Epoch [10/1000], Loss: 19178.2871
Epoch [10/1000], Loss: 16972.1055
Epoch [10/1000], Loss: 38054.0117
Epoch [20/1000], Loss: 15544.0967
Epoch [20/1000], Loss: 13297.3164
Epoch [20/1000], Loss: 16261.2295
Epoch [20/1000], Loss: 16484.2637
Epoch [20/1000], Loss: 22777.4863
Epoch [20/1000], Loss: 17500.2266
Epoch [20/1000], Loss: 18486.7402
Epoch [20/1000], Loss: 10297.1406
Epoch [30/1000], Loss: 15202.4883
Epoch [30/1000], Loss: 16322.585
Epoch [30/1000], Loss: 12750.0869
Epoch [30/1000], Loss: 17557.1855
Epoch [30/1000], Loss: 15262.5674
Epoch [30/1000], Loss: 13637.4951
Epoch [30/1000], Loss: 13482.4971
Epoch [30/1000], Loss: 7329.1968
Epoch [40/1000], Loss: 11901.2412
Epoch [40/1000], Loss: 12910.335
Epoch [40/1000], Loss: 15672.7695
Epoch [40/1000], Loss: 14174.4307
Epoch [40/1000], Loss: 8933.249
Epoch [40/1000], Loss

Increasing it to 1,000 epochs does give better results because the model has way more chances to learn, so the loss comes down a lot compared to before. It still jumps around a bit because it’s training in batches, but overall it’s clearly improving and getting more consistent.

## EXERCISE #2 (optional)
Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?


In [ ]:
import torch.nn as nn
import torch.optim as optim

# Define a new model with a different architecture
class DiabetesModel_V2(nn.Module):
    def __init__(self):
        super(DiabetesModel_V2, self).__init__()
        self.layers = nn.Sequential(
            # First layer: 10 input features, 8 neurons
            nn.Linear(10, 8),
            nn.ReLU(),
            # Second layer: 8 input features, 4 neurons
            nn.Linear(8, 4),
            nn.ReLU(),
            # Output layer: 4 input features, 1 output neuron
            nn.Linear(4, 1)
        )

    def forward(self, x):
        return self.layers(x)

# Initialize the new model, loss function, and optimizer
model = DiabetesModel_V2()
criterion = nn.MSELoss()
optimiser = optim.Adam(model.parameters(), lr=0.001)

print("New model architecture (DiabetesModel_V2) initialized.")
print(model)

New model architecture (DiabetesModel_V2) initialized.
DiabetesModel_V2(
  (layers): Sequential(
    (0): Linear(in_features=10, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
  )
)


No, there isn’t a single optimal architecture. The best way to get close is just to try different numbers of layers and neurons and see what gives the lowest error on the test data. If it’s too small it won’t learn enough, and if it’s too big it can overfit, so the best model is just the one that performs best overall.